# 03 — CLEAN → Mart | ISPRA Catasto Rifiuti

## Cosa fa questo notebook
A partire dal dataset **clean** (output del notebook 02), il notebook produce **tre output**:

1) **MART snapshot 2020-2023** (1 riga = 1 comune)
   - RD% 2020 e 2023, variazione in punti percentuali
   - RU totali 2020 e 2023 (t), variazione
   - RU pro capite 2020 e 2023 (kg/ab), variazione (supporto interpretativo)
   - classificazione in quadranti + flag RD↑ & RU↑ (risposta diretta alla domanda civica)
   - **`cluster_demografico`** (nuovo, v2) — fascia di popolazione 2023

2) **Serie storica 2019-2023** (formato long, 1 riga = comune-anno)
   - utile per una **seconda pagina Power BI** con trend e approfondimenti temporali

3) **Cluster Summary 2020-2023** (1 riga = cluster demografico) — NUOVO
   - tabella aggregata per la pagina Power BI "Performance per dimensione demografica"

## Output
- `data/mart/mart_comuni_delta_2020_2023_dashboard.parquet` *(aggiornato con `cluster_demografico`)*
- `data/mart/serie_comuni_rd_ru_2019_2023_powerbi_IT.parquet`
- `data/mart/cluster_summary_2020_2023.parquet` *(nuovo)*

Output Drive: https://drive.google.com/drive/folders/1Y1CCmyshifHTIQ1TT0jpNl-9C_HzLDgP?usp=drive_link

## Autori

- Matteo Cavo - Progettazione modello MART, indicatori, quadranti, cluster demografico
- Gabriele Sala - Revisione metodologica, validazione e integrazione nel progetto


In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import unicodedata

# --- GOOGLE COLAB SUPPORT ---
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Google Drive montato.")
except Exception:
    print("Esecuzione in ambiente locale (Drive non montato).")

# ======================
# PATHS (DataCivicLab / Lab standard)
# ======================
# Regola Lab: tutto sotto .../DataCivicLab/data/
#
# Colab (Drive-first): /content/drive/MyDrive/DataCivicLab/data
# Locale: imposta DCL_DATA_PATH oppure modifica BASE_PATH_LOCAL

BASE_PATH_DRIVE = Path("/content/drive/MyDrive/DataCivicLab/data")
BASE_PATH_LOCAL = Path("..").resolve() / "data"   # se notebook dentro /notebooks nella repo

# Override opzionale (consigliato): export DCL_DATA_PATH=".../DataCivicLab/data"
BASE_PATH = Path(os.getenv("DCL_DATA_PATH", str(BASE_PATH_DRIVE if IN_COLAB else BASE_PATH_LOCAL)))

# --- Directory standard ---
RAW_DIR   = BASE_PATH / "raw"
CLEAN_DIR = BASE_PATH / "clean"
MART_DIR  = BASE_PATH / "mart"

# --- Progetto specifico (qui: rifiuti) ---
MART_DOMAIN_DIR = MART_DIR / "ispra_catasto_rifiuti"
META_DIR = MART_DOMAIN_DIR / "_meta"

# --- File clean input (standard Lab) ---
CLEAN_FILE_DEFAULT = CLEAN_DIR / "ispra_catasto_rifiuti" / "ispra_catasto_rifiuti_clean.parquet"
CLEAN_FILE = Path(os.getenv("CLEAN_FILE", str(CLEAN_FILE_DEFAULT)))

# --- Output dir (default: mart/rifiuti) ---
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", str(MART_DOMAIN_DIR)))

# --- Creazione cartelle ---
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_PATH  :", BASE_PATH)
print("CLEAN_FILE :", CLEAN_FILE)
print("OUTPUT_DIR :", OUTPUT_DIR)
print("META_DIR   :", META_DIR)


Mounted at /content/drive
Google Drive montato.
BASE_PATH  : /content/drive/MyDrive/DataCivicLab/data
CLEAN_FILE : /content/drive/MyDrive/DataCivicLab/data/clean/ispra_catasto_rifiuti/ispra_catasto_rifiuti_clean.parquet
OUTPUT_DIR : /content/drive/MyDrive/DataCivicLab/data/mart/ispra_catasto_rifiuti
META_DIR   : /content/drive/MyDrive/DataCivicLab/data/mart/ispra_catasto_rifiuti/_meta


In [2]:
# ======================
# CARICA DATI PULITI
# ======================
df = pd.read_parquet(CLEAN_FILE)

print("Shape:", df.shape)
print("Colonne:", df.columns.tolist())
df.head()

Shape: (38631, 26)
Colonne: ['codice_comune_istat', 'regione', 'provincia', 'comune', 'popolazione', 'dato_riferito_a', 'frazione_umida1_t', 'verde_t', 'carta_e_cartone_t', 'vetro_t', 'legno_t', 'metallo_t', 'plastica_t', 'raee_t', 'tessili_t', 'selettiva_t', 'rifiuti_da_c_e_d_t', 'pulizia_stradale_a_recupero_t', 'ingombranti_misti_a_recupero_t', 'altro_t', 'raccolta_differenziata_tonnellate', 'ingombranti_a_smaltimento_t', 'indifferenziato_t', 'rifiuti_urbani_tonnellate', 'raccolta_differenziata_perc', 'anno']


,codice_comune_istat,regione,provincia,comune,popolazione,dato_riferito_a,frazione_umida1_t,verde_t,carta_e_cartone_t,vetro_t,...,rifiuti_da_c_e_d_t,pulizia_stradale_a_recupero_t,ingombranti_misti_a_recupero_t,altro_t,raccolta_differenziata_tonnellate,ingombranti_a_smaltimento_t,indifferenziato_t,rifiuti_urbani_tonnellate,raccolta_differenziata_perc,anno
0,01001001,Piemonte,Torino,AGLIE',2621.0,Comune,255.275,363.247,92.683,95.948,...,6.643,2.860,34.280,1.995,1007.864,NaN,549.186,1557.050,64.73,2019
1,01001002,Piemonte,Torino,AIRASCA,3598.0,Comune,174.489,140.355,239.377,100.730,...,10.801,7.760,110.507,NaN,1236.815,NaN,734.180,1970.995,62.75,2019
2,01001003,Piemonte,Torino,ALA DI STURA,441.0,Comune,3.240,13.450,23.636,26.368,...,0.159,NaN,30.587,0.005,112.624,NaN,172.960,285.584,39.44,2019
3,01001004,Piemonte,Torino,ALBIANO D'IVREA,1644.0,Comune,175.768,23.606,67.644,59.639,...,15.276,2.775,12.448,3.355,503.335,NaN,192.089,695.424,72.38,2019
4,01001006,Piemonte,Torino,ALMESE,6426.0,Comune,465.505,1237.986,352.661,268.956,...,64.082,63.484,96.568,5.621,2954.946,NaN,704.745,3659.691,80.74,2019


In [3]:
# ======================
# NORMALIZZAZIONI MINIME
# ======================

def norm_istat6(x):
    """Normalizza codice ISTAT a 6 cifre (string)."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(".0", "")
    s = "".join(ch for ch in s if ch.isdigit())
    if len(s) >= 8:
        s = s.zfill(8)[-6:]
    else:
        s = s.zfill(6)
    return s

def clean_comune_name(s):
    """Nome comune leggibile per Power BI (title case + unicode normalize)."""
    if pd.isna(s):
        return s
    s = str(s).strip()
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("'", "’")
    # se capita roba tipo '??' (encoding rotto), la eliminiamo
    s = s.replace("??", "")
    return s.title()

# ---- mapping colonne (supporta naming diversi) ----
# Cerca una colonna ISTAT
istat_col = None
for c in ["istat_comune_6", "codice_comune_istat", "istat_comune", "IstatComune"]:
    if c in df.columns:
        istat_col = c
        break
if istat_col is None:
    raise KeyError("Non trovo una colonna ISTAT (istat_comune_6 / codice_comune_istat / ...).")

# RU totali (t)
if "totale_ru_t" not in df.columns:
    if "rifiuti_urbani_tonnellate" in df.columns:
        df = df.rename(columns={"rifiuti_urbani_tonnellate": "totale_ru_t"})
    elif "Totale RU (t)" in df.columns:
        df = df.rename(columns={"Totale RU (t)": "totale_ru_t"})

# RD%
if "percentuale_rd" not in df.columns:
    if "raccolta_differenziata_perc" in df.columns:
        df = df.rename(columns={"raccolta_differenziata_perc": "percentuale_rd"})
    elif "Percentuale RD (%)" in df.columns:
        df = df.rename(columns={"Percentuale RD (%)": "percentuale_rd"})

needed = ["anno","comune","provincia","regione","popolazione","totale_ru_t","percentuale_rd"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Colonne mancanti nel CLEAN: {missing}")

df["istat_comune_6"] = df[istat_col].map(norm_istat6)
df["comune"] = df["comune"].map(clean_comune_name)

# RU pro capite (kg/abitante)
# ru_pro_capite_kg viene calcolato DOPO le aggregazioni (non qui)

df.head()


,codice_comune_istat,regione,provincia,comune,popolazione,dato_riferito_a,frazione_umida1_t,verde_t,carta_e_cartone_t,vetro_t,...,pulizia_stradale_a_recupero_t,ingombranti_misti_a_recupero_t,altro_t,raccolta_differenziata_tonnellate,ingombranti_a_smaltimento_t,indifferenziato_t,totale_ru_t,percentuale_rd,anno,istat_comune_6
0,01001001,Piemonte,Torino,Aglie’,2621.0,Comune,255.275,363.247,92.683,95.948,...,2.860,34.280,1.995,1007.864,NaN,549.186,1557.050,64.73,2019,001001
1,01001002,Piemonte,Torino,Airasca,3598.0,Comune,174.489,140.355,239.377,100.730,...,7.760,110.507,NaN,1236.815,NaN,734.180,1970.995,62.75,2019,001002
2,01001003,Piemonte,Torino,Ala Di Stura,441.0,Comune,3.240,13.450,23.636,26.368,...,NaN,30.587,0.005,112.624,NaN,172.960,285.584,39.44,2019,001003
3,01001004,Piemonte,Torino,Albiano D’Ivrea,1644.0,Comune,175.768,23.606,67.644,59.639,...,2.775,12.448,3.355,503.335,NaN,192.089,695.424,72.38,2019,001004
4,01001006,Piemonte,Torino,Almese,6426.0,Comune,465.505,1237.986,352.661,268.956,...,63.484,96.568,5.621,2954.946,NaN,704.745,3659.691,80.74,2019,001006


## Approfondimento: serie storica 2019–2023 (per Power BI)

Questo output **non sostituisce** il MART 2020–2023: lo affianca.
Serve per visualizzare l’evoluzione annuale di RD% e rifiuti (totali e pro capite),
per capire se i cambiamenti sono **graduali**, **discontinui** o legati a specifici anni.


In [4]:
# ======================
# SERIE STORICA 2019–2023 (LONG) — Esportazione  POWER BI
# ======================
YEARS_SERIES = [2019, 2020, 2021, 2022, 2023]

series = df[df["anno"].isin(YEARS_SERIES)].copy()

keys = ["istat_comune_6", "regione", "provincia", "comune"]

# consolidiamo eventuali duplicati (1 riga per comune-anno)
series = series.groupby(keys + ["anno"], as_index=False).agg({
    "popolazione": "sum",  # sum non mean (se ci sono duplicati vanno sommati)
    "totale_ru_t": "sum",
    "percentuale_rd": "mean",
})

# Calcola ru_pro_capite_kg DOPO l'aggregazione
series["ru_pro_capite_kg"] = (series["totale_ru_t"] * 1000) / series["popolazione"]

# rinomina colonne "friendly"
series = series.rename(columns={
    "percentuale_rd": "percentuale_rd",
    "totale_ru_t": "totale_ru_t",
    "ru_pro_capite_kg": "ru_pro_capite_kg",
})

# Rimuove NaN e infiniti (se ci sono) —> Power BI non li gestisce bene
series["rd_disponibile"] = series["percentuale_rd"].notna()
series["ru_disponibile"] = series["totale_ru_t"].notna()
series["percentuale_rd"] = series["percentuale_rd"].fillna(0)
series["totale_ru_t"] = series["totale_ru_t"].fillna(0)
series["ru_pro_capite_kg"] = series["ru_pro_capite_kg"].fillna(0)

# QA rapido serie
print("Serie storica righe:", series.shape[0])
print("Anni presenti:", sorted(series["anno"].unique().tolist()))
print("Duplicati (istat, anno):", int(series.duplicated(subset=["istat_comune_6","anno"]).sum()))
print("NaN% RD:", round(series["percentuale_rd"].isna().mean()*100, 2))
print("NaN% RU:", round(series["totale_ru_t"].isna().mean()*100, 2))

# ======================
# EXPORT SERIE STORICA 2019–2023 (Parquet)
# ======================

OUT_SERIE_PARQUET = os.path.join(
    MART_DOMAIN_DIR,
    "serie_comuni_rd_ru_2019_2023.parquet"
)

# sicurezza extra Power BI
series = series.replace([np.inf, -np.inf], np.nan).fillna(0)

# --- Parquet (riuso tecnico) ---
series.to_parquet(
    OUT_SERIE_PARQUET,
    index=False
)

# ======================
# META SERIE STORICA 2019–2023
# ======================

import json
from datetime import datetime

meta_series = {
    "dataset": "serie_comuni_rd_ru_2019_2023",
    "years_series": YEARS_SERIES,
    "generated_at": datetime.now().isoformat(),
    "input_file": str(CLEAN_FILE),
    "output_parquet": OUT_SERIE_PARQUET,
    "rows": int(series.shape[0]),
    "columns": list(series.columns)
}

META_SERIE_FILE = os.path.join(
    META_DIR,
    "serie_comuni_rd_ru_2019_2023.meta.json"
)

with open(META_SERIE_FILE, "w") as f:
    json.dump(meta_series, f, indent=4)

print("✅ Saved SERIE Parquet:", OUT_SERIE_PARQUET)
print("📝 Saved META SERIE:", META_SERIE_FILE)

series.head()


Serie storica righe: 38631
Anni presenti: [2019, 2020, 2021, 2022, 2023]
Duplicati (istat, anno): 0
NaN% RD: 0.0
NaN% RU: 0.0
✅ Saved SERIE Parquet: /content/drive/MyDrive/DataCivicLab/data/mart/ispra_catasto_rifiuti/serie_comuni_rd_ru_2019_2023.parquet
📝 Saved META SERIE: /content/drive/MyDrive/DataCivicLab/data/mart/ispra_catasto_rifiuti/_meta/serie_comuni_rd_ru_2019_2023.meta.json


,istat_comune_6,regione,provincia,comune,anno,popolazione,totale_ru_t,percentuale_rd,ru_pro_capite_kg,rd_disponibile,ru_disponibile
0,001001,Piemonte,Torino,Aglie’,2019,2621.0,1557.050,64.73,594.067150,True,True
1,001001,Piemonte,Torino,Aglie’,2020,2548.0,1563.893,66.64,613.772763,True,True
2,001001,Piemonte,Torino,Aglie’,2021,2549.0,1409.963,62.21,553.143586,True,True
3,001001,Piemonte,Torino,Aglie’,2022,2558.0,1415.399,65.68,553.322518,True,True
4,001001,Piemonte,Torino,Aglie’,2023,2603.0,1330.664,62.61,511.203995,True,True


In [5]:
# ======================
# FILTRO ANNI E AGGREGAZIONE (1 riga per comune-anno)
# ======================
Y0 = 2020
Y1 = 2023
d = df[df["anno"].isin([Y0, Y1])].copy()

keys = ["istat_comune_6","regione","provincia","comune"]

# In caso di duplicati (può capitare per frazioni / righe non-comune), consolidiamo
d = d.groupby(keys + ["anno"], as_index=False).agg({
    "popolazione": "sum",  # sum non mean
    "totale_ru_t": "sum",
    "percentuale_rd": "mean",
})

# Calcola ru_pro_capite_kg DOPO l'aggregazione
d["ru_pro_capite_kg"] = (d["totale_ru_t"] * 1000) / d["popolazione"]

print("Shape after groupby:", d.shape)
d.head()


Shape after groupby: (15479, 9)


,istat_comune_6,regione,provincia,comune,anno,popolazione,totale_ru_t,percentuale_rd,ru_pro_capite_kg
0,001001,Piemonte,Torino,Aglie’,2020,2548.0,1563.893,66.64,613.772763
1,001001,Piemonte,Torino,Aglie’,2023,2603.0,1330.664,62.61,511.203995
2,001002,Piemonte,Torino,Airasca,2020,3569.0,2032.627,61.69,569.522836
3,001002,Piemonte,Torino,Airasca,2023,3686.0,1806.090,72.24,489.986435
4,001003,Piemonte,Torino,Ala Di Stura,2020,448.0,369.621,38.50,825.046875


In [6]:
# ======================
# WIDE 2020 vs 2023 + DELTA
# ======================
w = d.pivot_table(
    index=keys,
    columns="anno",
    values=["percentuale_rd","totale_ru_t","ru_pro_capite_kg","popolazione"],
    aggfunc="first"
).reset_index()

# flatten columns
w.columns = [
    f"{a}_{b}" if isinstance(b, (int, np.integer)) else a
    for a, b in w.columns
]

# rename convenience
w = w.rename(columns={
    f"percentuale_rd_{Y0}": f"percentuale_rd_{Y0}",
    f"percentuale_rd_{Y1}": f"percentuale_rd_{Y1}",
    f"totale_ru_t_{Y0}": f"totale_ru_t_{Y0}",
    f"totale_ru_t_{Y1}": f"totale_ru_t_{Y1}",
    f"ru_pro_capite_kg_{Y0}": f"ru_pro_capite_kg_{Y0}",
    f"ru_pro_capite_kg_{Y1}": f"ru_pro_capite_kg_{Y1}",
    f"popolazione_{Y0}": f"popolazione_{Y0}",
    f"popolazione_{Y1}": f"popolazione_{Y1}",
})

# deltas
w["delta_rd_pp"] = w[f"percentuale_rd_{Y1}"] - w[f"percentuale_rd_{Y0}"]
w["delta_ru_totali_t"] = w[f"totale_ru_t_{Y1}"] - w[f"totale_ru_t_{Y0}"]
w["delta_ru_pro_capite"] = w[f"ru_pro_capite_kg_{Y1}"] - w[f"ru_pro_capite_kg_{Y0}"]

w.head()


,istat_comune_6,regione,provincia,comune,percentuale_rd_2020,percentuale_rd_2023,popolazione_2020,popolazione_2023,ru_pro_capite_kg_2020,ru_pro_capite_kg_2023,totale_ru_t_2020,totale_ru_t_2023,delta_rd_pp,delta_ru_totali_t,delta_ru_pro_capite
0,001001,Piemonte,Torino,Aglie’,66.64,62.61,2548.0,2603.0,613.772763,511.203995,1563.893,1330.664,-4.03,-233.229,-102.568768
1,001002,Piemonte,Torino,Airasca,61.69,72.24,3569.0,3686.0,569.522836,489.986435,2032.627,1806.090,10.55,-226.537,-79.536400
2,001003,Piemonte,Torino,Ala Di Stura,38.50,44.92,448.0,473.0,825.046875,732.786469,369.621,346.608,6.42,-23.013,-92.260406
3,001004,Piemonte,Torino,Albiano D’Ivrea,70.82,80.69,1650.0,1619.0,411.358788,437.741198,678.742,708.703,9.87,29.961,26.382410
4,001006,Piemonte,Torino,Almese,79.49,77.32,6448.0,6323.0,596.065602,568.449154,3843.431,3594.304,-2.17,-249.127,-27.616448


In [7]:
# ======================
# CLASSIFICAZIONE (quadranti) + FLAG DOMANDA CIVICA
# ======================

def quadrante(delta_rd_pp, delta_ru_tot):
    if pd.isna(delta_rd_pp) or pd.isna(delta_ru_tot):
        return "Dati mancanti"
    if delta_rd_pp > 0 and delta_ru_tot <= 0:
        return "Virtuosi (RD↑, RU↓)"
    if delta_rd_pp > 0 and delta_ru_tot > 0:
        return "Migliora RD ma aumenta RU (RD↑, RU↑)"
    if delta_rd_pp <= 0 and delta_ru_tot <= 0:
        return "Riduce RU ma non RD (RD↓, RU↓)"
    return "Peggiora entrambi (RD↓, RU↑)"

w["quadrante"] = w.apply(lambda r: quadrante(r["delta_rd_pp"], r["delta_ru_totali_t"]), axis=1)

# Flag che risponde direttamente alla domanda civica:
w["rd_su_rifiuti_su"] = (w["delta_rd_pp"] > 0) & (w["delta_ru_totali_t"] > 0)

# (Opzionale) Virtuoso strutturale: RD↑ e RU pro capite ↓
w["virtuoso_strutturale"] = (w["delta_rd_pp"] > 0) & (w["delta_ru_pro_capite"] < 0)

w[["comune","delta_rd_pp","delta_ru_totali_t","rd_su_rifiuti_su","quadrante"]].head(10)


,comune,delta_rd_pp,delta_ru_totali_t,rd_su_rifiuti_su,quadrante
0,Aglie’,-4.03,-233.229,False,"Riduce RU ma non RD (RD↓, RU↓)"
1,Airasca,10.55,-226.537,False,"Virtuosi (RD↑, RU↓)"
2,Ala Di Stura,6.42,-23.013,False,"Virtuosi (RD↑, RU↓)"
3,Albiano D’Ivrea,9.87,29.961,True,"Migliora RD ma aumenta RU (RD↑, RU↑)"
4,Almese,-2.17,-249.127,False,"Riduce RU ma non RD (RD↓, RU↓)"
5,Alpette,-1.38,1.821,False,"Peggiora entrambi (RD↓, RU↑)"
6,Alpignano,2.15,559.728,True,"Migliora RD ma aumenta RU (RD↑, RU↑)"
7,Andezeno,4.69,-32.255,False,"Virtuosi (RD↑, RU↓)"
8,Andrate,5.40,3.301,True,"Migliora RD ma aumenta RU (RD↑, RU↑)"
9,Angrogna,-0.58,0.045,False,"Peggiora entrambi (RD↓, RU↑)"


In [8]:
# ======================
# DEDUPLICAZIONE (1 riga per istat_comune_6)
# ======================
# In alcuni casi ISPRA può avere più righe con lo stesso codice ISTAT.
# Per il MART Power BI vogliamo 1 riga = 1 comune.
# Regola: teniamo la riga con popolazione 2023 più alta (più "affidabile" nei merge).

before = len(w)

pop_col = f"popolazione_{Y1}"
if pop_col not in w.columns:
    # fallback: se manca (non dovrebbe), deduplica senza criterio
    w_dedup = w.drop_duplicates(subset=["istat_comune_6"], keep="first").copy()
else:
    w_dedup = (
        w.sort_values(by=["istat_comune_6", pop_col], ascending=[True, False])
         .drop_duplicates(subset=["istat_comune_6"], keep="first")
         .reset_index(drop=True)
    )

after = len(w_dedup)
print("Righe prima:", before)
print("Righe dopo :", after)
print("Duplicati rimossi:", before - after)

# Da qui in avanti usiamo SEMPRE w_final
w_final = w_dedup


Righe prima: 7797
Righe dopo : 7779
Duplicati rimossi: 18


In [9]:
# ======================
# QA MINIMO
# ======================
print("Comuni (righe):", len(w_final))
print("Quota RD↑ & RU↑ (totali):", round(w_final["rd_su_rifiuti_su"].mean()*100, 3), "%")
print("N RD↑ & RU↑:", int(w_final["rd_su_rifiuti_su"].sum()))
print("N virtuosi strutturali (RD↑ & RU_pc↓):", int(w_final["virtuoso_strutturale"].sum()))

# Duplicati chiave
dup = w_final.duplicated(subset=["istat_comune_6"]).sum()
print("Duplicati per istat_comune_6:", int(dup))

# Range RD
print("RD% min/max 2020:", float(w_final[f"percentuale_rd_{Y0}"].min()), float(w_final[f"percentuale_rd_{Y0}"].max()))
print("RD% min/max 2023:", float(w_final[f"percentuale_rd_{Y1}"].min()), float(w_final[f"percentuale_rd_{Y1}"].max()))


Comuni (righe): 7779
Quota RD↑ & RU↑ (totali): 30.737 %
N RD↑ & RU↑: 2391
N virtuosi strutturali (RD↑ & RU_pc↓): 1913
Duplicati per istat_comune_6: 0
RD% min/max 2020: 0.18 100.0
RD% min/max 2023: 0.06 99.92


In [10]:
# ======================
# EXPORT MART DELTA 2020–2023 (Drive)
# ======================

out_cols = [
    "istat_comune_6","regione","provincia","comune",
    f"percentuale_rd_{Y0}", f"percentuale_rd_{Y1}", "delta_rd_pp",
    f"totale_ru_t_{Y0}", f"totale_ru_t_{Y1}", "delta_ru_totali_t",
    f"ru_pro_capite_kg_{Y0}", f"ru_pro_capite_kg_{Y1}", "delta_ru_pro_capite",
    "rd_su_rifiuti_su","virtuoso_strutturale","quadrante"
]

mart = w_final[out_cols].copy()

# Sicurezza Power BI
mart = mart.replace([np.inf, -np.inf], np.nan).fillna(0)

OUT_DASHBOARD_PARQUET = os.path.join(
    MART_DOMAIN_DIR,
    "mart_comuni_delta_2020_2023_dashboard.parquet"
)

# --- Parquet (riuso tecnico / BigQuery / analytics) ---
mart.to_parquet(
    OUT_DASHBOARD_PARQUET,
    index=False
)

# ======================
# META MART DELTA 2020–2023
# ======================

META_DIR = os.path.join(MART_DOMAIN_DIR, "_meta")
os.makedirs(META_DIR, exist_ok=True)

meta_mart = {
    "dataset": "mart_comuni_delta_2020_2023_dashboard",
    "years_delta": [Y0, Y1],
    "generated_at": datetime.now().isoformat(),
    "input_file": str(CLEAN_FILE),
    "output_parquet": OUT_DASHBOARD_PARQUET,
    "rows": int(mart.shape[0]),
    "columns": list(mart.columns)
}

META_MART_FILE = os.path.join(
    META_DIR,
    "mart_comuni_delta_2020_2023_dashboard.meta.json"
)

with open(META_MART_FILE, "w") as f:
    json.dump(meta_mart, f, indent=4)

print("✅ Saved MART Parquet:", OUT_DASHBOARD_PARQUET)
print("📝 Saved META MART:", META_MART_FILE)

mart.head()


✅ Saved MART Parquet: /content/drive/MyDrive/DataCivicLab/data/mart/ispra_catasto_rifiuti/mart_comuni_delta_2020_2023_dashboard.parquet
📝 Saved META MART: /content/drive/MyDrive/DataCivicLab/data/mart/ispra_catasto_rifiuti/_meta/mart_comuni_delta_2020_2023_dashboard.meta.json


,istat_comune_6,regione,provincia,comune,percentuale_rd_2020,percentuale_rd_2023,delta_rd_pp,totale_ru_t_2020,totale_ru_t_2023,delta_ru_totali_t,ru_pro_capite_kg_2020,ru_pro_capite_kg_2023,delta_ru_pro_capite,rd_su_rifiuti_su,virtuoso_strutturale,quadrante
0,001001,Piemonte,Torino,Aglie’,66.64,62.61,-4.03,1563.893,1330.664,-233.229,613.772763,511.203995,-102.568768,False,False,"Riduce RU ma non RD (RD↓, RU↓)"
1,001002,Piemonte,Torino,Airasca,61.69,72.24,10.55,2032.627,1806.090,-226.537,569.522836,489.986435,-79.536400,False,True,"Virtuosi (RD↑, RU↓)"
2,001003,Piemonte,Torino,Ala Di Stura,38.50,44.92,6.42,369.621,346.608,-23.013,825.046875,732.786469,-92.260406,False,True,"Virtuosi (RD↑, RU↓)"
3,001004,Piemonte,Torino,Albiano D’Ivrea,70.82,80.69,9.87,678.742,708.703,29.961,411.358788,437.741198,26.382410,True,False,"Migliora RD ma aumenta RU (RD↑, RU↑)"
4,001006,Piemonte,Torino,Almese,79.49,77.32,-2.17,3843.431,3594.304,-249.127,596.065602,568.449154,-27.616448,False,False,"Riduce RU ma non RD (RD↓, RU↓)"


## Estensione: Analisi per Dimensione Demografica

Questa sezione aggiunge `cluster_demografico` al mart e produce `cluster_summary_2020_2023.csv`,
un file aggregato pronto per la pagina Power BI **"Performance per dimensione demografica"**.

La popolazione di riferimento e' quella **2023** (gia' presente nel mart come `popolazione_{Y1}`,
derivata dal pivot wide effettuato sopra). Non occorre nessun join esterno.

| Cluster | Fascia abitativa |
|---------|------------------|
| `<5k` | < 5.000 ab |
| `5k-20k` | 5.000 - 19.999 ab |
| `20k-100k` | 20.000 - 99.999 ab |
| `>100k` | >= 100.000 ab |

> **Nota metodologica**: la classificazione usa la popolazione 2023 (anno finale),
> coerente con gli indicatori di stato (RD%, RU pro capite). I delta 2020->2023 non dipendono da questa.


In [11]:
# ======================
# CLUSTER DEMOGRAFICO
# Basato su popolazione_{Y1} da w_final (il mart esclude le colonne popolazione)
# ======================

assert f'popolazione_{Y1}' in w_final.columns, (
    f"Colonna 'popolazione_{Y1}' non trovata in w_final.")

CLUSTER_ORDER = ['<5k', '5k-20k', '20k-100k', '>100k']

def assegna_cluster(pop):
    if pd.isna(pop) or pop == 0:
        return 'N/D'
    elif pop < 5_000:
        return '<5k'
    elif pop < 20_000:
        return '5k-20k'
    elif pop < 100_000:
        return '20k-100k'
    else:
        return '>100k'

# Ricaviamo il cluster da w_final e lo mergiamo nel mart
pop_cluster = w_final[['istat_comune_6', f'popolazione_{Y1}']].copy()
pop_cluster['cluster_demografico'] = pop_cluster[f'popolazione_{Y1}'].apply(assegna_cluster)

mart = mart.merge(
    pop_cluster[['istat_comune_6', f'popolazione_{Y1}', 'cluster_demografico']],
    on='istat_comune_6',
    how='left'
)

print('=== Distribuzione cluster_demografico ===')
vc = mart['cluster_demografico'].value_counts()
for label in CLUSTER_ORDER:
    print(f'  {label:>12}: {vc.get(label, 0):>5} comuni')
nd = vc.get('N/D', 0)
if nd:
    print(f'  {"N/D":>12}: {nd:>5} comuni  <- da verificare')

print('\nNaN in cluster_demografico:', mart['cluster_demografico'].isna().sum())
print('Righe mart (deve restare invariato):', len(mart))

=== Distribuzione cluster_demografico ===
           <5k:  5390 comuni
        5k-20k:  1859 comuni
      20k-100k:   467 comuni
         >100k:    44 comuni
           N/D:    19 comuni  <- da verificare

NaN in cluster_demografico: 0
Righe mart (deve restare invariato): 7779


In [12]:
# ======================
# QA CLUSTER DEMOGRAFICO
# ======================

# 1. Zero NaN
nan_count = mart['cluster_demografico'].isna().sum()
assert nan_count == 0, f'ERRORE: {nan_count} NaN in cluster_demografico'
print('OK  NaN in cluster_demografico: 0')

# 2. Categorie attese
cats = set(mart['cluster_demografico'].unique())
expected = {'<5k', '5k-20k', '20k-100k', '>100k'}
unexpected = cats - expected - {'N/D'}
assert not unexpected, f'Categorie inattese: {unexpected}'
print(f'OK  Categorie: {sorted(cats)}')

# 3. Delta pre-esistenti non alterati
for col in ['delta_rd_pp', 'delta_ru_totali_t', 'delta_ru_pro_capite']:
    n_nan = mart[col].isna().sum()
    print(f'    {col}: {n_nan} NaN')

# 4. Totale comuni invariato
print(f'OK  Totale comuni: {len(mart)}')
print('\nQA cluster demografico completato')


OK  NaN in cluster_demografico: 0
OK  Categorie: ['20k-100k', '5k-20k', '<5k', '>100k', 'N/D']
    delta_rd_pp: 0 NaN
    delta_ru_totali_t: 0 NaN
    delta_ru_pro_capite: 0 NaN
OK  Totale comuni: 7779

QA cluster demografico completato


In [13]:
# ======================
# CLUSTER SUMMARY — tabella aggregata per Power BI
# ======================

cluster_summary = (
    mart[mart['cluster_demografico'].isin(CLUSTER_ORDER)]
    .groupby('cluster_demografico', observed=True)
    .agg(
        n_comuni          = ('istat_comune_6',        'count'),
        rd_media_2023     = ('percentuale_rd_2023',   'mean'),
        ru_pc_medio_2023  = ('ru_pro_capite_kg_2023', 'mean'),
        delta_rd_pp_medio = ('delta_rd_pp',           'mean'),
        delta_ru_pc_medio = ('delta_ru_pro_capite',   'mean'),
        pct_rd_su_ru_su   = ('rd_su_rifiuti_su',      'mean'),
        pct_virtuosi      = ('virtuoso_strutturale',   'mean'),
    )
    .reset_index()
)

# Ordine logico asse X Power BI
cluster_summary['cluster_order'] = cluster_summary['cluster_demografico'].map(
    {c: i for i, c in enumerate(CLUSTER_ORDER)}
)
cluster_summary = cluster_summary.sort_values('cluster_order').drop(columns='cluster_order')

# Arrotondamento
float_cols = cluster_summary.select_dtypes('float').columns
cluster_summary[float_cols] = cluster_summary[float_cols].round(2)

cluster_summary


,cluster_demografico,n_comuni,rd_media_2023,ru_pc_medio_2023,delta_rd_pp_medio,delta_ru_pc_medio,pct_rd_su_ru_su,pct_virtuosi
2,<5k,5390,67.74,452.06,3.18,7.45,0.30,0.25
1,5k-20k,1859,74.07,487.85,2.80,7.68,0.33,0.23
0,20k-100k,467,68.99,492.08,3.60,4.72,0.34,0.26
3,>100k,44,61.09,534.25,4.50,-1.47,0.32,0.27


In [14]:
# ======================
# DISTRIBUZIONE QUADRANTI x CLUSTER — per matrix Power BI
# ======================

cluster_quadranti = (
    mart[mart['cluster_demografico'].isin(CLUSTER_ORDER)]
    .groupby(['cluster_demografico', 'quadrante'], observed=True)
    .size()
    .reset_index(name='n_comuni')
)

# Pivot leggibile
pivot_q = cluster_quadranti.pivot(
    index='cluster_demografico', columns='quadrante', values='n_comuni'
).fillna(0).astype(int)
pivot_q = pivot_q.loc[[c for c in CLUSTER_ORDER if c in pivot_q.index]]

print(pivot_q.to_string())


quadrante            Dati mancanti  Migliora RD ma aumenta RU (RD↑, RU↑)  Peggiora entrambi (RD↓, RU↑)  Riduce RU ma non RD (RD↓, RU↓)  Virtuosi (RD↑, RU↓)
cluster_demografico                                                                                                                                        
<5k                            100                                  1607                           969                            1202                 1512
5k-20k                           5                                   609                           377                             420                  448
20k-100k                         0                                   161                            88                              97                  121
>100k                            0                                    14                             5                               9                   16


In [15]:
# ======================
# EXPORT — mart v2 (con cluster_demografico) + cluster_summary
# ======================

# Colonne output v2 (aggiunge cluster_demografico)
out_cols_v2 = [
    'istat_comune_6', 'regione', 'provincia', 'comune',
    f'percentuale_rd_{Y0}', f'percentuale_rd_{Y1}', 'delta_rd_pp',
    f'totale_ru_t_{Y0}',    f'totale_ru_t_{Y1}',   'delta_ru_totali_t',
    f'ru_pro_capite_kg_{Y0}', f'ru_pro_capite_kg_{Y1}', 'delta_ru_pro_capite',
    f'popolazione_{Y1}',
    'rd_su_rifiuti_su', 'virtuoso_strutturale', 'quadrante',
    'cluster_demografico',
]

mart_v2 = mart[out_cols_v2].copy()
mart_v2 = mart_v2.replace([np.inf, -np.inf], np.nan).fillna(0)

OUT_MART_V2_PARQUET = os.path.join(MART_DOMAIN_DIR, 'mart_comuni_delta_2020_2023_dashboard.parquet')
OUT_CLUSTER_PARQUET = os.path.join(MART_DOMAIN_DIR, 'cluster_summary_2020_2023.parquet')

mart_v2.to_parquet(OUT_MART_V2_PARQUET, index=False)
cluster_summary.to_parquet(OUT_CLUSTER_PARQUET, index=False)

# Meta aggiornata
meta_v2 = {
    'dataset'         : 'mart_comuni_delta_2020_2023_dashboard',
    'version_note'    : 'v2 - Added cluster_demografico (popolazione_2023); closes #32',
    'years_delta'     : [Y0, Y1],
    'generated_at'    : datetime.now().isoformat(),
    'rows'            : int(mart_v2.shape[0]),
    'columns'         : list(mart_v2.columns),
    'cluster_summary' : OUT_CLUSTER_PARQUET,
}
META_V2_FILE = os.path.join(META_DIR, 'mart_comuni_delta_2020_2023_dashboard.meta.json')
with open(META_V2_FILE, 'w') as f:
    json.dump(meta_v2, f, indent=4)

print('✅ Mart v2 Parquet    :', OUT_MART_V2_PARQUET)
print('✅ Cluster summary Parquet:', OUT_CLUSTER_PARQUET)
print('📝 Meta aggiornata   :', META_V2_FILE)
print('\nColonne mart v2:', list(mart_v2.columns))


✅ Mart v2 Parquet    : /content/drive/MyDrive/DataCivicLab/data/mart/ispra_catasto_rifiuti/mart_comuni_delta_2020_2023_dashboard.parquet
✅ Cluster summary Parquet: /content/drive/MyDrive/DataCivicLab/data/mart/ispra_catasto_rifiuti/cluster_summary_2020_2023.parquet
📝 Meta aggiornata   : /content/drive/MyDrive/DataCivicLab/data/mart/ispra_catasto_rifiuti/_meta/mart_comuni_delta_2020_2023_dashboard.meta.json

Colonne mart v2: ['istat_comune_6', 'regione', 'provincia', 'comune', 'percentuale_rd_2020', 'percentuale_rd_2023', 'delta_rd_pp', 'totale_ru_t_2020', 'totale_ru_t_2023', 'delta_ru_totali_t', 'ru_pro_capite_kg_2020', 'ru_pro_capite_kg_2023', 'delta_ru_pro_capite', 'popolazione_2023', 'rd_su_rifiuti_su', 'virtuoso_strutturale', 'quadrante', 'cluster_demografico']


# Analisi Rifiuti Urbani e Raccolta Differenziata (ISPRA 2020–2023)

## Domanda Civica

**Ci sono comuni che migliorano la raccolta differenziata ma aumentano i rifiuti totali?**

**Sì.** I dati ISPRA a livello comunale mostrano che il miglioramento della raccolta differenziata non implica automaticamente una riduzione dei rifiuti prodotti.

---

## Risultati Principali (2020–2023)

| Indicatore | Valore |
|------------|--------|
| Comuni analizzati | **7.779** |
| Comuni con RD in aumento e rifiuti totali in aumento | **2.391 (30,7%)** |
| Comuni virtuosi strutturali (RD aumenta, RU pro capite diminuisce) | **1.913 (24,6%)** |
| RD media 2020 → 2023 | **65,5% → 69,1% (+3,1 pp)** |
| RU pro capite media 2020 → 2023 | **452 → 462 kg/ab (+7,3 kg/ab)** |

---

## Classificazione dei comuni per quadrante

| Quadrante | Comuni |
|----------|--------|
| Migliora RD ma aumenta RU (RD↑, RU↑) | **2.391** |
| Virtuosi (RD↑, RU↓) | **2.097** |
| Riduce RU ma non RD (RD↓, RU↓) | **1.728** |
| Peggiora entrambi (RD↓, RU↑) | **1.439** |
| Dati mancanti | **124** |

---

##  Note Metodologiche

- **Perché il confronto 2020–2023:** il confronto inizio–fine su un orizzonte triennale offre un segnale stabile per la classificazione dei comuni, riducendo il rumore dato da oscillazioni annuali.
- **Filtro dati aggregati:** il dataset esclude i record ISPRA con `Dato riferito a` diverso da `"Comune"`. Per i dettagli vedere la documentazione in fondo al notebook `02_raw_clean`.
- **Valori mancanti:** la copertura dei dati nella serie storica è completa per le variabili principali (RD%, tonnellate totali, pro capite). I **124 comuni con dati mancanti** nel dataset delta sono esclusi dalla classificazione per quadrante.

---

## Output Prodotti

| File | Descrizione |
|------|-------------|
| `serie_comuni_rd_ru_2019_2023.parquet` | Dataset long (comune × anno), serie storica completa |
| `mart_comuni_delta_2020_2023.parquet` | Dataset wide con delta e classificazione per comune |
| `cluster_summary_2020_2023.parquet` | Aggregato per cluster demografico (parquet) |

---

## Estensione: Cluster Demografico (v2)

Il mart include ora `cluster_demografico`, derivato dalla popolazione 2023 del comune,
che classifica ogni comune in quattro fasce: `<5k`, `5k-20k`, `20k-100k`, `>100k`.

Il file `cluster_summary_2020_2023` contiene le metriche aggregate per fascia
(RD media, RU pro capite, delta medi, % comuni problematici e virtuosi).

I 19 comuni con `cluster_demografico = N/D` presentano `popolazione_2023 = NaN`
nella fonte ISPRA e sono esclusi dalle aggregazioni per cluster ma inclusi nel mart principale.

---

## Lettura dei risultati

Circa **1 comune su 3** aumenta la raccolta differenziata ma produce più rifiuti complessivi.  
I comuni realmente virtuosi, che differenziano di più e riducono anche i rifiuti pro capite, sono il **24,6%** del totale.

Il dato suggerisce che politiche basate esclusivamente sull'aumento della raccolta differenziata non incidono necessariamente sulla quantità di rifiuti prodotti: **prevenzione e riduzione dei consumi** restano centrali. La raccolta differenziata è uno strumento necessario, ma non risolutivo.
